In [3]:
from sqlalchemy import create_engine, text
from pathlib import Path
import pandas as pd

USERNAME = "postgres"
PASSWORD = "admin123"
HOST     = "localhost"
PORT     = "5434"
DATABASE = "DigitalTransformation"

engine = create_engine(
    f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

print("Database connection successful")

Database connection successful


In [4]:
# =============================================================
# QUERY FUNCTIONS — POSTGRESQL
# =============================================================

def get_stations():
    """Get all stations"""
    with engine.connect() as conn:
        result = conn.execute(text("SELECT * FROM stations"))
        return pd.DataFrame(result.fetchall(), columns=result.keys())

def get_by_city(city):
    """Get all readings for a specific city"""
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT r.*, s.city, s.state, s.station_name
            FROM weather_readings r
            JOIN stations s ON r.station_id = s.station_id
            WHERE LOWER(s.city) = LOWER(:city)
        """), {"city": city})
        return pd.DataFrame(result.fetchall(), columns=result.keys())

def get_by_month(year, month):
    """Get all readings for a specific month"""
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT r.*, s.city, s.state
            FROM weather_readings r
            JOIN stations s ON r.station_id = s.station_id
            WHERE EXTRACT(YEAR FROM r.timestamp) = :year
            AND EXTRACT(MONTH FROM r.timestamp) = :month
        """), {"year": year, "month": month})
        return pd.DataFrame(result.fetchall(), columns=result.keys())

def get_pollutant(pollutant, city=None):
    """Get readings for a specific pollutant, optionally filtered by city"""
    query = """
        SELECT p.*, s.city, s.state
        FROM pollutant_readings p
        JOIN stations s ON p.station_id = s.station_id
        WHERE LOWER(p.pollutant) = LOWER(:pollutant)
    """
    params = {"pollutant": pollutant}
    if city:
        query += " AND LOWER(s.city) = LOWER(:city)"
        params["city"] = city

    with engine.connect() as conn:
        result = conn.execute(text(query), params)
        return pd.DataFrame(result.fetchall(), columns=result.keys())

def get_avg_pollutant_by_month(pollutant):
    """Get monthly average for a pollutant"""
    with engine.connect() as conn:
        result = conn.execute(text("""
            SELECT 
                EXTRACT(YEAR FROM timestamp) as year,
                EXTRACT(MONTH FROM timestamp) as month,
                AVG(value) as avg_value,
                MAX(value) as max_value,
                MIN(value) as min_value
            FROM pollutant_readings
            WHERE LOWER(pollutant) = LOWER(:pollutant)
            GROUP BY year, month
            ORDER BY year, month
        """), {"pollutant": pollutant})
        return pd.DataFrame(result.fetchall(), columns=result.keys())

def get_wide_month(year, month):
    """Load a specific month from the wide table"""
    path = Path(f'wide_table/wide_{year}_{month:02d}.csv')
    if path.exists():
        return pd.read_csv(path)
    else:
        print(f"No wide table file found for {year}-{month:02d}")
        return None

# --- Test the functions ---
print("Stations:")
print(get_stations())

print("\nJanuary 2025 readings:")
print(get_by_month(2025, 1).head())

print("\npm25 readings:")
print(get_pollutant('pm25').head())

Stations:
  station_id  state   city                  station_name
0   site_104  Delhi  Delhi  Burari Crossing, Delhi - IMD
1   site_108  Delhi  Delhi        Aya Nagar, Delhi - IMD
2  site_1420  Delhi  Delhi     Ashok Vihar, Delhi - DPCC
3  site_1560  Delhi  Delhi          Bawana, Delhi - DPCC
4   site_301  Delhi  Delhi     Anand Vihar, Delhi - DPCC
5  site_5024  Delhi  Delhi          Alipur, Delhi - DPCC

January 2025 readings:
  station_id                 timestamp       at_c  rh_percent    ws_m_s  \
0   site_104 2025-01-01 00:00:00+03:00  25.761743   63.614581  1.286361   
1   site_104 2025-01-01 00:15:00+03:00  25.761743   63.614581  1.286361   
2   site_104 2025-01-01 00:30:00+03:00  25.761743   63.614581  1.286361   
3   site_104 2025-01-01 00:45:00+03:00  25.761743   63.614581  1.286361   
4   site_104 2025-01-01 01:00:00+03:00  25.761743   63.614581  1.286361   

       wd_deg     rf_mm    sr_w_mt2     bp_mmhg   city  state  
0  150.758626  0.033738  125.007555  930.175799  Del

In [6]:
!pip install dash dash-bootstrap-components plotly

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 346.0 kB/s eta 0:00:23
   -- ------------------------------------- 0.5/8.4 MB 346.0 kB/s eta 0:00:23
   -- ------------------------------------- 0.5/8.4 MB 346.0 kB/s eta 0:00:23
   --- ------------------------------------ 0.8/8.4 MB 366.0 kB/s eta 0:00:21
   --- ------------------------------------ 0.8/8.4 MB 366.0 kB/s e

In [12]:
# --- Load pollutant data ---
with engine.connect() as conn:
    pollutants_data = pd.DataFrame(
        conn.execute(text("""
            SELECT p.*, s.city, s.state,
                   EXTRACT(YEAR FROM p.timestamp) as year,
                   EXTRACT(MONTH FROM p.timestamp) as month
            FROM pollutant_readings p
            JOIN stations s ON p.station_id = s.station_id
        """)).fetchall(),
        columns=['station_id', 'timestamp', 'pollutant',
                 'value', 'city', 'state', 'year', 'month']
    )

    weather_data = pd.DataFrame(
        conn.execute(text("""
            SELECT r.*, s.city
            FROM weather_readings r
            JOIN stations s ON r.station_id = s.station_id
        """)).fetchall(),
        columns=['station_id', 'timestamp', 'at_c', 'rh_percent',
                 'ws_m_s', 'wd_deg', 'rf_mm', 'sr_w_mt2',
                 'bp_mmhg', 'city']
    )

print(f"Pollutant data: {len(pollutants_data):,} rows")
print(f"Weather data: {len(weather_data):,} rows")

Pollutant data: 3,495,959 rows
Weather data: 409,427 rows


In [13]:
import dash
from dash import dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([

    # Header
    dbc.Row([
        dbc.Col(html.H1("DigiTransformation — Air Quality Dashboard",
            className="text-center my-4 text-white",
            style={'backgroundColor': '#2c3e50', 'padding': '20px',
                   'borderRadius': '10px'}))
    ]),

    # Filters
    dbc.Row([
        dbc.Col([
            html.Label("Pollutant:"),
            dcc.Dropdown(
                id='pollutant-dropdown',
                options=[{'label': p.upper(), 'value': p}
                         for p in sorted(pollutants_data['pollutant'].unique())],
                value='pm25',
                clearable=False
            )
        ], width=4),
        dbc.Col([
            html.Label("Station:"),
            dcc.Dropdown(
                id='station-dropdown',
                options=[{'label': s, 'value': s}
                         for s in sorted(pollutants_data['station_id'].unique())],
                value=None,
                placeholder="All stations",
                clearable=True
            )
        ], width=4),
        dbc.Col([
            html.Label("Year:"),
            dcc.Dropdown(
                id='year-dropdown',
                options=[{'label': str(int(y)), 'value': int(y)}
                         for y in sorted(pollutants_data['year'].unique())],
                value=int(pollutants_data['year'].max()),
                clearable=False
            )
        ], width=4),
    ], className="my-3"),

    # Summary cards
    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H6("Total Stations", className="card-title"),
            html.H3(id='total-stations', className="text-primary")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H6("Avg Value", className="card-title"),
            html.H3(id='avg-value', className="text-success")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H6("Max Value", className="card-title"),
            html.H3(id='max-value', className="text-danger")
        ])), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H6("Total Readings", className="card-title"),
            html.H3(id='total-readings', className="text-info")
        ])), width=3),
    ], className="my-3"),

    # Charts row 1
    dbc.Row([
        dbc.Col([
            html.H5("Monthly Trend"),
            dcc.Graph(id='trend-chart')
        ], width=8),
        dbc.Col([
            html.H5("Avg by Station"),
            dcc.Graph(id='station-chart')
        ], width=4),
    ], className="my-3"),

    # Charts row 2
    dbc.Row([
        dbc.Col([
            html.H5("Temperature vs Pollutant"),
            dcc.Graph(id='scatter-chart')
        ], width=6),
        dbc.Col([
            html.H5("Pollutant Distribution"),
            dcc.Graph(id='box-chart')
        ], width=6),
    ], className="my-3"),

    # Data table
    dbc.Row([
        dbc.Col([
            html.H5("Raw Data Table"),
            dash_table.DataTable(
                id='data-table',
                page_size=15,
                sort_action='native',
                filter_action='native',
                style_table={'overflowX': 'auto'},
                style_cell={'textAlign': 'left', 'padding': '8px'},
                style_header={
                    'backgroundColor': '#2c3e50',
                    'color': 'white',
                    'fontWeight': 'bold'
                },
                style_data_conditional=[{
                    'if': {'row_index': 'odd'},
                    'backgroundColor': '#f8f9fa'
                }]
            )
        ])
    ], className="my-3")

], fluid=True)


@app.callback(
    [Output('total-stations', 'children'),
     Output('avg-value', 'children'),
     Output('max-value', 'children'),
     Output('total-readings', 'children'),
     Output('trend-chart', 'figure'),
     Output('station-chart', 'figure'),
     Output('scatter-chart', 'figure'),
     Output('box-chart', 'figure'),
     Output('data-table', 'data'),
     Output('data-table', 'columns')],
    [Input('pollutant-dropdown', 'value'),
     Input('station-dropdown', 'value'),
     Input('year-dropdown', 'value')]
)
def update_dashboard(pollutant, station, year):

    # Filter pollutant data
    filtered = pollutants_data[
        (pollutants_data['pollutant'] == pollutant) &
        (pollutants_data['year'] == year)
    ]
    if station:
        filtered = filtered[filtered['station_id'] == station]

    # Filter weather data
    weather_filtered = weather_data.copy()
    if station:
        weather_filtered = weather_filtered[
            weather_filtered['station_id'] == station
        ]

    # Summary cards
    total_stations = str(filtered['station_id'].nunique())
    avg_val        = f"{filtered['value'].mean():.2f}"
    max_val        = f"{filtered['value'].max():.2f}"
    total_readings = f"{len(filtered):,}"

    # Monthly trend
    monthly = filtered.groupby('month')['value'].mean().reset_index()
    monthly['month'] = monthly['month'].astype(int)
    trend_fig = px.line(
        monthly, x='month', y='value',
        title=f"Monthly Avg {pollutant.upper()} — {year}",
        markers=True,
        color_discrete_sequence=['#3498db']
    )

    # Station bar chart
    station_avg = filtered.groupby('station_id')['value'].mean().reset_index()
    station_fig = px.bar(
        station_avg, x='station_id', y='value',
        title=f"Avg {pollutant.upper()} by Station",
        color_discrete_sequence=['#27ae60']
    )

    # Scatter — temperature vs pollutant
    merged = filtered.merge(
        weather_filtered[['station_id', 'timestamp', 'at_c']],
        on=['station_id', 'timestamp'],
        how='inner'
    ).dropna(subset=['at_c'])

    scatter_fig = px.scatter(
        merged.sample(min(1000, len(merged))),
        x='at_c', y='value',
        color='station_id',
        title=f"Temperature vs {pollutant.upper()}",
        labels={'at_c': 'Temperature (°C)', 'value': pollutant.upper()}
    )

    # Box plot
    box_fig = px.box(
        filtered, x='station_id', y='value',
        title=f"{pollutant.upper()} Distribution by Station",
        color_discrete_sequence=['#e74c3c']
    )

    # Data table
    table_data = filtered[['station_id', 'city', 'timestamp',
                            'pollutant', 'value']]\
        .head(200).to_dict('records')
    table_cols = [{'name': c, 'id': c} for c in
                  ['station_id', 'city', 'timestamp', 'pollutant', 'value']]

    return (total_stations, avg_val, max_val, total_readings,
            trend_fig, station_fig, scatter_fig, box_fig,
            table_data, table_cols)


app.run(debug=True, port=8050) #go to http://localhost:8050 for full dashboard web app

In [10]:
import plotly.express as px
import plotly.io as pio

# Example — PM25 monthly trend
fig = px.line(
    monthly_data, 
    x='month', 
    y='value',
    title='Monthly PM25 Trend'
)

# Save as standalone HTML file
pio.write_html(fig, 'dashboard.html')
print("Dashboard saved as dashboard.html")

Dashboard saved as dashboard.html
